# 🚀 Optuna Hyperparameter Optimization - Google Colab

**Configurazione**: modifica i file in `src/config/` prima di eseguire.

**File da modificare**:
- `src/config/tuning_config.py` → `MODEL_NAME`, `N_TRIALS`, `STUDY_NAME`, etc.
- `src/config/training_config.py` → `NAIVE_MAE_*`, `EPOCHS`

## 1. 📦 Setup

In [ ]:
# Monta Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Cartella risultati su Drive
import os
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/optuna_results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"📁 Risultati: {DRIVE_RESULTS_DIR}")

In [ ]:
# Clone/Update repository (branch develop)
REPO_PATH = '/content/Progetto_deep_learning'
BRANCH = 'develop'

if not os.path.exists(REPO_PATH):
    print(f"🔄 Clonando branch {BRANCH}...")
    !git clone -b {BRANCH} https://github.com/scorzaluca/Progetto_deep_learning.git
else:
    print(f"📂 Repository esistente, aggiorno branch {BRANCH}...")
    !cd {REPO_PATH} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

# Setup Python path
%pip install optuna -q
import sys
sys.path.insert(0, REPO_PATH)

print("\n✅ Setup completato!")
print("\n📝 ORA MODIFICA I FILE DI CONFIG PRIMA DI CONTINUARE:")
print(f"   {REPO_PATH}/src/config/tuning_config.py")
print(f"   {REPO_PATH}/src/config/training_config.py")

## ⚠️ MODIFICA I FILE DI CONFIG ORA

1. Clicca l'icona **cartella** 📁 a sinistra
2. Naviga a: `Progetto_deep_learning/src/config/`
3. Doppio click su `tuning_config.py` per aprirlo
4. Modifica i parametri e **salva** (Ctrl+S)
5. **Riesegui la cella sotto** per ricaricare i moduli
6. Poi continua con le celle successive

In [ ]:
# ⚡ RIESEGUI QUESTA CELLA DOPO OGNI MODIFICA AI FILE DI CONFIG ⚡

import importlib
import src.config.tuning_config as tc
import src.config.training_config as trc
import src.config.model_config as mc
import src.config as cfg
import src.DataLoading as dl

# Reload tutti i moduli config
importlib.reload(tc)
importlib.reload(trc)
importlib.reload(mc)
importlib.reload(cfg)
importlib.reload(dl)


# Importa configurazione aggiornata
from src.config import (
    SEED, TARGET_COL,
    MODEL_NAME, N_TRIALS, N_FOLDS, TUNING_EPOCHS, PATIENCE,
    STUDY_NAME, NEW_STUDY,
    NAIVE_MAE_FINAL_FOLD, EPOCHS
)

# Verifica configurazione
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"📊 Modello: {MODEL_NAME}")
print(f"📝 Studio: {STUDY_NAME} ({'NUOVO' if NEW_STUDY else 'RIPRENDE'})")
print(f"🔢 Trials: {N_TRIALS}, Folds: {N_FOLDS}")
print(f"⏱️ Epochs: {TUNING_EPOCHS}, Patience: {PATIENCE}")
print(f"🖥️ Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

## 2. 📊 Carica Dati

In [ ]:
from src.Utils import set_seed, load_data_and_folds

DATA_PATH = f"{REPO_PATH}/data/processed/preprocessed_ds.csv"
STORAGE_PATH = f"{DRIVE_RESULTS_DIR}/optuna_studies.db"

set_seed(SEED)
df, folds = load_data_and_folds(data_path=DATA_PATH)

print(f"\n📈 Dataset: {df.shape}")
print(f"📂 Folds: {len(folds)}")
print(f"💾 Database: {STORAGE_PATH}")

## 3. 🔍 Ottimizzazione Optuna

In [ ]:
from src.Tuning import OptunaOptimizer

optimizer = OptunaOptimizer(
    model_name=MODEL_NAME,
    folds=folds,
    device=DEVICE,
    config={
        "n_trials": N_TRIALS,
        "n_folds": N_FOLDS,
        "epochs": TUNING_EPOCHS,
        "patience": PATIENCE,
    },
    storage_path=STORAGE_PATH,
    verbose=False,
)

result = optimizer.optimize(
    study_name=STUDY_NAME,
    n_trials=N_TRIALS,
    new_study=NEW_STUDY,
)

print(f"\n✅ Best MASE: {result['best_mase']:.4f}")
print(f"Best params: {result['best_params']}")

## 4. 📈 Visualizza Risultati

In [ ]:
import optuna
import matplotlib.pyplot as plt

study = optuna.load_study(study_name=STUDY_NAME, storage=f"sqlite:///{STORAGE_PATH}")

values = [t.value for t in study.trials if t.value is not None]
best_values = [min(values[:i+1]) for i in range(len(values))]

plt.figure(figsize=(10, 4))
plt.plot(values, 'o-', alpha=0.6, label='Trial MASE')
plt.plot(best_values, 'r-', linewidth=2, label='Best MASE')
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Trial')
plt.ylabel('MASE')
plt.title(f'{MODEL_NAME.upper()} - Optimization History')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(f"{DRIVE_RESULTS_DIR}/{STUDY_NAME}_plot.png", dpi=150)
plt.show()

## 5. 🏋️ Final Training (Opzionale)

In [ ]:
from src.Training.engine import create_model, fit_model
from src.DataLoading import create_final_train_val_loaders
from src.config import LOOKBACK

best_params = result["best_params"]
train_loader, val_loader, scaler = create_final_train_val_loaders(df, TARGET_COL, LOOKBACK)

model = create_model(MODEL_NAME, best_params)
model.to(DEVICE)

model, history, best_epoch = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=best_params.get("lr", 0.001),
    device=DEVICE,
    patience=10,
    optimizer_kwargs={"weight_decay": best_params.get("weight_decay", 0.001)},
    grad_clip_norm=best_params.get("grad_clip_norm", 1.0),
    verbose=True,
    baseline_mae=NAIVE_MAE_FINAL_FOLD,
)

print(f"\n✅ Best MASE finale: {min(history['val_mase']):.4f}")

## 6. 💾 Salva Risultati

In [ ]:
import json

torch.save(model.state_dict(), f"{DRIVE_RESULTS_DIR}/{STUDY_NAME}_model.pth")

with open(f"{DRIVE_RESULTS_DIR}/{STUDY_NAME}_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

with open(f"{DRIVE_RESULTS_DIR}/{STUDY_NAME}_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"✅ Salvato in: {DRIVE_RESULTS_DIR}/")
!ls -la {DRIVE_RESULTS_DIR}/

### 7. ⚓ Visualizza studi nel db sul drive

In [ ]:
import optuna
STORAGE_PATH="/content/drive/MyDrive/optuna_results/optuna_studies.db"
# List all available studies
all_studies = optuna.study.get_all_study_names(storage=f"sqlite:///{STORAGE_PATH}")
print("Available studies:")
for name in all_studies:
    print(f"  - {name}")